# 1 — Run the experiment

Produces `outputs/results{suffix}.parquet`, the single input to notebooks 2, 3 and 4.

Pipeline: controlled fact insertion → exact-memory evaluation → compression exposure →
repeated probing → scored trials.

**Read this before running.** The research doc commits to shortening the inference
sliding window so that target facts actually cross into compressed memory. A merged
AHN checkpoint loads through a stock `Qwen2ForCausalLM` carrying Qwen2.5's own, much
larger window — if it is not forced, nothing is ever compressed and the run measures
prompt length instead of memory. `models.load` forces it and prints what the
checkpoint reported; the `window_is_exceeded` gate at the bottom fails the run if no
trial cleared it. This is blocker #1 in `protocol/open_decisions.md`.

Pilot mode verifies plumbing on a GPU (local CUDA or Colab). Its numbers are never
cited as results.

**Where to put the AHN repo:** local → `vendor/AHN` (or `$AHN_REPO`); Colab →
`/content/AHN`. `config.ahn_repo()` picks the first that exists.

In [ ]:
# SKIP this cell if you already ran:  bash scripts/setup_gpu.sh
# Otherwise: kerne---------------------------------------------------------------------------

import shutil, subprocess, sys
from pathlib import Path

def run(cmd):
    print("+", " ".join(map(str, cmd))); subprocess.check_call(cmd)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / "pyproject.toml").exists()), here)
AHN = Path("/content/AHN") if IN_COLAB else ROOT / "vendor" / "AHN"
py = sys.executable
print("python:", py)

if IN_COLAB:
    def install(*a):
        run([py, "-m", "pip", "install", *a])
else:
    if ".venv" not in py:
        raise SystemExit("Wrong kernel. Select: Python (ahn-mdc / uv)")
    uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
    if not Path(uv).exists():
        raise SystemExit("uv not found")
    def install(*a):
        run([uv, "pip", "install", "--python", py, *a])

if not (AHN / "examples/scripts/utils/merge_weights.py").is_file():
    if AHN.exists():
        shutil.rmtree(AHN)
    AHN.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", "https://github.com/ByteDance-Seed/AHN.git", str(AHN)])
else:
    print("AHN ok:", AHN)

install("git+https://github.com/Seerkfang/flash-linear-attention.git@main")
install("git+https://github.com/Seerkfang/LLaMA-Factory.git@main")
install("flash-attn==2.8.3", "--extra-index-url", "https://wheels.astral.sh/simple/cu128/")
install("-e", str(AHN))
print("DONE — restart kernel")


python: /home/dd3tech/ahn-mdc/.venv/bin/python
AHN ok: /home/dd3tech/ahn-mdc/vendor/AHN
+ /home/dd3tech/.local/bin/uv pip install --python /home/dd3tech/ahn-mdc/.venv/bin/python git+https://github.com/Seerkfang/flash-linear-attention.git@main


Using Python 3.11.14 environment at: /home/dd3tech/ahn-mdc/.venv
Resolved 67 packages in 889ms
Uninstalled 2 packages in 9ms
Installed 2 packages in 11ms
 - datasets==3.2.0
 + datasets==5.0.1
 - fsspec==2026.7.0
 + fsspec==2026.6.0
Using Python 3.11.14 environment at: /home/dd3tech/ahn-mdc/.venv


+ /home/dd3tech/.local/bin/uv pip install --python /home/dd3tech/ahn-mdc/.venv/bin/python git+https://github.com/Seerkfang/LLaMA-Factory.git@main


Resolved 134 packages in 949ms
Uninstalled 6 packages in 70ms
Installed 6 packages in 19ms
 - datasets==5.0.1
 + datasets==3.2.0
 - fsspec==2026.6.0
 + fsspec==2024.9.0
 - markupsafe==3.0.3
 + markupsafe==2.1.5
 - pandas==3.0.5
 + pandas==2.3.3
 - pillow==12.3.0
 + pillow==11.3.0
 - tokenizers==0.21.4
 + tokenizers==0.21.0
Using Python 3.11.14 environment at: /home/dd3tech/ahn-mdc/.venv


+ /home/dd3tech/.local/bin/uv pip install --python /home/dd3tech/ahn-mdc/.venv/bin/python flash-attn==2.8.3 --extra-index-url https://wheels.astral.sh/simple/cu128/


Resolved 31 packages in 935ms
Uninstalled 9 packages in 154ms
Installed 9 packages in 145ms
 - cuda-toolkit==13.0.3.0
 + cuda-toolkit==13.0.2
 - nvidia-cublas==13.1.1.3
 + nvidia-cublas==13.1.0.3
 - nvidia-cudnn-cu13==9.20.0.48
 + nvidia-cudnn-cu13==9.19.0.56
 - nvidia-cusparselt-cu13==0.8.1
 + nvidia-cusparselt-cu13==0.8.0
 - nvidia-nccl-cu13==2.29.7
 + nvidia-nccl-cu13==2.28.9
 - nvidia-nvjitlink==13.3.33
 + nvidia-nvjitlink==13.0.88
 - setuptools==84.0.0
 + setuptools==81.0.0
 - torch==2.13.0
 + torch==2.11.0
 - triton==3.7.1
 + triton==3.6.0
Using Python 3.11.14 environment at: /home/dd3tech/ahn-mdc/.venv
Resolved 1 package in 1ms
   Building ahn @ file:///home/dd3tech/ahn-mdc/vendor/AHN


+ /home/dd3tech/.local/bin/uv pip install --python /home/dd3tech/ahn-mdc/.venv/bin/python -e /home/dd3tech/ahn-mdc/vendor/AHN
DONE — restart kernel


      Built ahn @ file:///home/dd3tech/ahn-mdc/vendor/AHN
Prepared 1 package in 519ms
Uninstalled 1 package in 0.65ms
Installed 1 package in 0.79ms
 ~ ahn==0.1.0 (from file:///home/dd3tech/ahn-mdc/vendor/AHN)


In [2]:
import importlib
import sys
from pathlib import Path

import importlib.metadata as _md
assert _md.version("transformers").startswith("4.51"), (
    f"transformers={_md.version('transformers')} — need 4.51.0. "
    "In a terminal: uv pip install --python .venv/bin/python 'transformers==4.51.0' "
    "then Restart kernel."
)
import torch
import transformers
print("transformers", transformers.__version__)

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / "config").exists()), here)
sys.path.insert(0, str(ROOT / "src"))

import ahnexp.config as config
import ahnexp.dataset as dataset
import ahnexp.evaluate as evaluate
import ahnexp.models as models
import ahnexp.report as report

# Jupyter keeps the first import in memory. Reload so file edits take effect
# without Restart Kernel.
for module in (config, dataset, evaluate, models, report):
    importlib.reload(module)
config.clear_caches()

MODE = "pilot"          # "pilot" | "full"
LENGTH_MATCHED = False  # trade filler for pressure to hold total prompt length constant
# Optional override; otherwise: $AHN_REPO → /content/AHN (Colab) → vendor/AHN
AHN_REPO = None  # e.g. Path("/content/AHN") or Path("../vendor/AHN")

settings = config.run_mode(MODE)
RAW = config.output_path("raw", MODE)
AHN_REPO = config.ahn_repo(AHN_REPO)

assert torch.cuda.is_available(), "Need a CUDA GPU for this notebook (CPU is analysis-only)."
print("cuda:", torch.cuda.get_device_name(0))
print("colab:", config.is_colab())
print("ahn_repo:", AHN_REPO)
print("dataset.py:", dataset.__file__)
print("collision check uses word boundaries:", "_leaks_answer" in dir(dataset))
print(settings)
print("window forced to:", config.experiment()["models"]["sliding_window"]["force"])
print("raw output:", RAW.relative_to(ROOT))


transformers 4.51.0
cuda: NVIDIA RTX 2000 Ada Generation Laptop GPU
colab: False
ahn_repo: /home/dd3tech/ahn-mdc/vendor/AHN
dataset.py: /home/dd3tech/ahn-mdc/src/ahnexp/dataset.py
collision check uses word boundaries: True
{'purpose': 'pipeline verification only — never cited as a result', 'n_items': 10, 'seeds': [0], 'grid': 'pilot_grid', 'hardware': 'gpu', 'suffix': '_pilot'}
window forced to: 256
raw output: outputs/results_pilot.parquet


## Items

Generated once, from a fixed seed, and replayed byte-identically by every arm. That is
what makes the comparison paired: fact difficulty cancels out of every contrast.

`assert_no_collision` runs on each item — a distractor containing the target's answer
would let the model score correct without retrieving anything from memory.

In [3]:
import pandas as pd

ITEM_SEED = 0
items = dataset.generate_items(n_items=settings["n_items"], seed=ITEM_SEED)
ITEMS_PATH = config.output_path("items", MODE)
dataset.save_items(items, ITEMS_PATH, seed=ITEM_SEED)
print(f"{len(items)} items -> {ITEMS_PATH.relative_to(ROOT)}")
print(pd.Series([i.fact.fact_type for i in items]).value_counts().to_string())
print()
print(items[0].fact.text, "|", items[0].fact.question, "->", items[0].fact.answer)

10 items -> outputs/items_pilot.json
numerical           2
temporal            2
entity-attribute    2
multi-hop           2
contradictory       2

Person_0's employee ID is 100000. | What is Person_0's employee ID? -> 100000


## Run the grid

Arms load one at a time and are released before the next — three merged 3B checkpoints
do not co-exist on a typical single GPU. Merging happens on first use and is cached in
`merged_ckpt/`.

`assert_matched` runs at the end and raises if the arms differ in anything other than
architecture and AHN parameter count. If it raises, the comparison is not valid and no
amount of downstream statistics fixes it.

Set `arms=["gated_deltanet"]` for a first pass. H1 and H3 need one arm; H2's
architecture comparison needs all four.

In [4]:
results = evaluate.run_grid(
    items,
    tokenizer_for_items=None,
    ahn_repo=AHN_REPO,
    mode=MODE,
    arms=["gated_deltanet"],                 # None = every arm in config
    length_matched=LENGTH_MATCHED,
)
results.to_parquet(RAW, index=False)
print(f"{len(results)} trials -> {RAW.relative_to(ROOT)}")
results.groupby(["architecture", "memory_condition"])["correct"].agg(["mean", "size"])

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

checkpoint reports sliding_window=256, use_sliding_window=False -> forcing 256


/home/dd3tech/ahn-mdc/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/dd3tech/ahn-mdc/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/dd3tech/ahn-mdc/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/dd3tech/ahn-mdc/.venv/lib/python3.11/site-pa

50 trials -> outputs/results_pilot.parquet


mean  size
architecture   memory_condition            
gated_deltanet exact_memory       1.0    20
               recurrent_memory   0.2    30

## Gates

Nothing downstream is trustworthy until these pass.

- `window_is_exceeded` — did anything actually get compressed? A `FAIL` means the run
  measured prompt length, not memory.
- `exact_memory_accuracy` — the control condition must land in 70–80% (85% acceptable).
  A `RED_FLAG` at 90% means existing models already solve the task and the
  fact/distractor design has to be hardened before spending A40 time.
- `threshold_locked` — stays `BLOCKED` until the H2 threshold has a published source.
  Expected until decision #3 closes; it blocks H2's numbers, not H1 or H3.
- `matched_design`, `min_cell_size` — balance and power.

In [5]:
gates = report.gate_report(results)
blocking = report.blocking(gates)
print(gates.to_string(index=False))
print(f"\n{len(blocking)} blocking issue(s)")
if len(blocking):
    print(blocking.to_string(index=False))

                   gate          scope  verdict                                                                detail
  exact_memory_accuracy gated_deltanet RED_FLAG 100.0% >= 90%: task already solved, harden the fact/distractor design
     window_is_exceeded         design     PASS                      30/50 trials past the 256-token window (max 797)
       threshold_locked             H2  BLOCKED                                          H2 threshold T is not locked
threshold_clears_window             H2     PASS                                     T=768 tokens = 3.0 windows of 256
          min_cell_size            all     FAIL                                                 35 underpowered cells
         matched_design            all     PASS                                                    1 arm(s), balanced

3 blocking issue(s)
                 gate          scope  verdict                                                                detail
exact_memory_accuracy gated_deltanet 